In [1]:
# =================================================================================================
# Raw data : This is All Lab data from NHANES
# Previously Done : To run this file, the list of CSVs below have processed datasets yearly into saperate notebooks
# Purpose : This Aggregator program will get all yearly csvs and create one dataset file 2003-2014 
# Specification : Each NHANE dataset has SEQN "Participent ID" which is uniqud id for labs. 
# Files Used :    "neighbourhood_nhanes_labs_2003_2004.csv","neighbourhood_nhanes_labs_2005_2006.csv","neighbourhood_nhanes_labs_2007_2008.csv","neighbourhood_nhanes_labs_2009_2010.csv",
# "neighbourhood_nhanes_labs_2011_2012.csv",
# "neighbourhood_nhanes_labs_2013_2014.csv"
# Data Availability Matrix : https://docs.google.com/spreadsheets/d/1E_zIzHZsEs0o-XWx8HbTN7ZPODge7EEC/edit?gid=1721397761#gid=1721397761
# Source Site : https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2003/DataFiles/L10AM_C.htm
# Developer Notes :  The notebook had created for windows desktop environment with Python3.
# File Creator : Neha Khairnar
# ===================================================================================================

import pandas as pd
from pathlib import Path

# =========================
# 📂 DATA LOCATION
# =========================
DATA_DIR = Path("../data/processed")

files = [
    "neighbourhood_nhanes_labs_2003_2004.csv",
    "neighbourhood_nhanes_labs_2005_2006.csv",
    "neighbourhood_nhanes_labs_2007_2008.csv",
    "neighbourhood_nhanes_labs_2009_2010.csv",
    "neighbourhood_nhanes_labs_2011_2012.csv",
    "neighbourhood_nhanes_labs_2013_2014.csv"
]

# =========================
# 📦 STORAGE
# =========================
all_dfs = []
audit_rows = []

# =========================
# 🔁 PROCESS EACH FILE
# =========================
for file in files:
    
    path = DATA_DIR / file
    df = pd.read_csv(path)
    
    # -------------------------
    # 🗓 cycle label
    # -------------------------
    cycle = file.replace("NHANES_LABS_", "").replace(".csv", "")
    df["NHANES_CYCLE"] = cycle
    
    # -------------------------
    # 📊 BASIC STATS
    # -------------------------
    rows = df.shape[0]
    cols = df.shape[1]
    missing_total = df.isna().sum().sum()
    missing_pct = (missing_total / (rows * cols)) * 100
    
    # -------------------------
    # 🧾 COLUMN LIST
    # -------------------------
    columns = list(df.columns)
    
    # -------------------------
    # ❌ TOP MISSING COLUMNS
    # -------------------------
    missing_by_col = df.isna().sum().sort_values(ascending=False)
    
    top_missing = "; ".join(
        [f"{col}:{val}" for col, val in missing_by_col.head(10).items()]
    )
    
    # -------------------------
    # 📦 STORE DATASET
    # -------------------------
    all_dfs.append(df)
    
    # -------------------------
    # 🧾 AUDIT ROW
    # -------------------------
    audit_rows.append({
        "file": file,
        "cycle": cycle,
        "rows": rows,
        "columns": cols,
        "missing_total": missing_total,
        "missing_percent": round(missing_pct, 2),
        "column_count": len(columns),
        "top_missing_columns": top_missing
    })

# =========================
# 🌎 FINAL APPEND (MASTER DATASET)
# =========================
final_df = pd.concat(all_dfs, ignore_index=True)

# =========================
# 🧹 OPTIONAL STANDARD FLAGS
# =========================
final_df["start_year"] = final_df["NHANES_CYCLE"].str.split("_").str[0]
final_df["end_year"] = final_df["NHANES_CYCLE"].str.split("_").str[1]

# =========================
# 📊 AUDIT DATAFRAME
# =========================
audit_df = pd.DataFrame(audit_rows)

# =========================
# 💾 SAVE FILES
# =========================
output_master = DATA_DIR / "NHANES_LABS_MASTER_2003_2014.csv"
output_audit = DATA_DIR / "NHANES_LABS_AUDIT_REPORT.csv"

final_df.to_csv(output_master, index=False)
audit_df.to_csv(output_audit, index=False)

# =========================
# 📋 SUMMARY PRINT
# =========================
print("\n" + "="*80)
print("✅ NHANES MASTER PIPELINE COMPLETE")
print("MASTER DATASET:")
print("Rows:", final_df.shape[0])
print("Columns:", final_df.shape[1])
print("Unique SEQN:", final_df["SEQN"].nunique())

print("\nAUDIT REPORT:")
print(audit_df)

print("\n💾 Saved files:")
print("📦 MASTER:", output_master)
print("📊 AUDIT :", output_audit)


✅ NHANES MASTER PIPELINE COMPLETE
MASTER DATASET:
Rows: 56139
Columns: 46
Unique SEQN: 56139

AUDIT REPORT:
                                      file  \
0  neighbourhood_nhanes_labs_2003_2004.csv   
1  neighbourhood_nhanes_labs_2005_2006.csv   
2  neighbourhood_nhanes_labs_2007_2008.csv   
3  neighbourhood_nhanes_labs_2009_2010.csv   
4  neighbourhood_nhanes_labs_2011_2012.csv   
5  neighbourhood_nhanes_labs_2013_2014.csv   

                                 cycle  rows  columns  missing_total  \
0  neighbourhood_nhanes_labs_2003_2004  9179       36         140427   
1  neighbourhood_nhanes_labs_2005_2006  9440       39         158637   
2  neighbourhood_nhanes_labs_2007_2008  9307       39         155161   
3  neighbourhood_nhanes_labs_2009_2010  9835       39         161479   
4  neighbourhood_nhanes_labs_2011_2012  8956       39         148748   
5  neighbourhood_nhanes_labs_2013_2014  9422       37         140972   

   missing_percent  column_count  \
0            42.50         